# M7 · Backfill `fact_maintenance` (header) + `fact_maintenance_line`

**Goal:** load vehicle-grain maintenance work-orders and their line items (offense / sinistre / reparation) into the warehouse for the BI maintenance dashboard.

**SQL files:**
- `sql/18_fact_maintenance_incr.sql` (header — UPSERT on (tenant_id, id_maintenance))
- `sql/19_fact_maintenance_line_incr.sql` (lines — DELETE-INSERT-on-window)

**Run order matters:** header first (lines reference parent `id_maintenance`).

In [ ]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.config import settings, load_pipeline_config
from accent_fleet.db import get_engine
from sqlalchemy import text
from datetime import datetime, timedelta

s = settings()
cfg = load_pipeline_config()
MIN_TIME = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
print(f'DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}')

## 2. Inputs

In [ ]:
with get_engine().connect() as conn:
    end_time = conn.execute(text('SELECT MAX(date_operation) FROM staging.maintenance')).scalar_one()
    counts = {
        'maintenance':  conn.execute(text('SELECT COUNT(*) FROM staging.maintenance')).scalar_one(),
        'offense':      conn.execute(text('SELECT COUNT(*) FROM staging.offense')).scalar_one(),
        'sinistre':     conn.execute(text('SELECT COUNT(*) FROM staging.sinistre')).scalar_one(),
        'reparation':   conn.execute(text('SELECT COUNT(*) FROM staging.reparation')).scalar_one(),
    }
print('staging.maintenance max date_operation =', end_time)
print('row counts:', counts)

## 3. Execute

In [ ]:
import importlib, time, pandas as pd
import accent_fleet.transforms.facts as facts_module
from accent_fleet.pipeline.run_log import begin_run, end_run
facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

run_id = begin_run(mode='notebook:09_load_fact_maintenance')
progress, cursor, t0 = [], MIN_TIME, time.time()
try:
    while cursor < end_time:
        nxt = min(cursor + timedelta(days=CHUNK_DAYS), end_time)
        # Header MUST run before lines for the same window.
        h = load_fact_incremental('fact_maintenance',      run_id=run_id, window_end=nxt)
        l = load_fact_incremental('fact_maintenance_line', run_id=run_id, window_end=nxt)
        progress.append({'window_start': cursor, 'window_end': nxt,
                         'header_rows': h.rows_loaded, 'line_rows': l.rows_loaded})
        cursor = nxt
    end_run(run_id, status='success', rows_loaded=sum(p['header_rows']+p['line_rows'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc)); raise
print(f'done in {time.time()-t0:.1f}s')
pd.DataFrame(progress).tail(10)

## 4. Inspect

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    h_total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_maintenance')).scalar_one()
    l_total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_maintenance_line')).scalar_one()
    line_mix = pd.read_sql(text('''SELECT line_type, COUNT(*) AS n
                                    FROM warehouse.fact_maintenance_line
                                    GROUP BY 1 ORDER BY 2 DESC'''), conn)
    cost = pd.read_sql(text('''SELECT
        COUNT(*) FILTER (WHERE total_cost > 0) AS rows_with_cost,
        AVG(total_cost) FILTER (WHERE total_cost > 0) AS avg_cost,
        SUM(total_cost) AS total_cost FROM warehouse.fact_maintenance'''), conn)
print(f'fact_maintenance rows: {h_total:,}  fact_maintenance_line rows: {l_total:,}')
display(line_mix); display(cost)